Now as we build our warehouse, we can use it to build some data marts for specific business insights. 

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum

In [4]:
spark = (
    SparkSession.builder
    .appName("Customer_Churn_BI")
    .enableHiveSupport()
    .getOrCreate()
)

2026-09-04 20:27:23,543 WARN util.Utils: Your hostname, localhost.localdomain resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
2026-09-04 20:27:23,548 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
2026-09-04 20:27:25,048 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
HIVE_DB = "customer_churn_db"

In [6]:
spark.sql(f"USE {HIVE_DB}")

print("Current database:")
spark.sql("SELECT current_database()").show()

2026-09-04 20:31:12,745 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-09-04 20:31:12,746 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist
2026-09-04 20:31:16,155 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


Current database:


+------------------+
|current_database()|
+------------------+
| customer_churn_db|
+------------------+



In [7]:
spark.sql("SHOW TABLES").show()

+-----------------+--------------------+-----------+
|         database|           tableName|isTemporary|
+-----------------+--------------------+-----------+
|customer_churn_db|     customer_silver|      false|
|customer_churn_db|customer_usage_si...|      false|
|customer_churn_db|        dim_customer|      false|
|customer_churn_db|            dim_date|      false|
|customer_churn_db|fact_customer_mon...|      false|
|customer_churn_db|       offers_silver|      false|
|customer_churn_db|      tickets_silver|      false|
+-----------------+--------------------+-----------+



# Insight 1 — Early Warning Churn Signals
In my data we don't have churn date so we don't know the behaviour of the customer in his last month but we have is_churned that indicate whether the customer is churned or not, so we will use this to study the behaviour and the engagements of the customer with whether this behaviours attend him to leave the bank or not.

That is by identifying:

- Monthly balance changes
- Monthly product changes
- Whether the customer eventually churned

And that's by:
## 1. First join between my fact table and dimensions

In [18]:
monthly_behavior_df = spark.sql("""
    SELECT
        f.cust_key,
        c.customer_id,
        d.full_date AS activity_month,

        c.is_churned,

        f.ending_monthly_balance,
        f.active_products_count

    FROM fact_customer_monthly_activity_df f

    JOIN dim_customer c
        ON f.cust_key = c.cust_key

    JOIN dim_date d
        ON f.date_key = d.date_key
""")

In [19]:
monthly_behavior_df.show(20, truncate=False)

+--------+-----------+--------------+----------+----------------------+---------------------+
|cust_key|customer_id|activity_month|is_churned|ending_monthly_balance|active_products_count|
+--------+-----------+--------------+----------+----------------------+---------------------+
|219     |15571284   |2026-07-01    |false     |35254.14              |2                    |
|822     |15586996   |2026-07-01    |false     |246.76                |1                    |
|1142    |15594594   |2026-07-01    |false     |88098.72              |1                    |
|2799    |15636330   |2026-07-01    |true      |null                  |0                    |
|2843    |15637131   |2026-07-01    |false     |614.64                |1                    |
|4061    |15667460   |2026-07-01    |false     |346.02                |1                    |
|5156    |15694506   |2026-07-01    |false     |53614.82              |1                    |
|7176    |15745399   |2026-07-01    |true      |911.36      

## 2. Calculate the previous month values (product counts and balance) and that's by using the window function lag()

In [20]:
behavior_window = (Window.partitionBy("customer_id").orderBy("activity_month"))

last_known_balance_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("activity_month")
    .rowsBetween(Window.unboundedPreceding, -1)
)

monthly_behavior_df = (
    monthly_behavior_df
    .withColumn(
        "previous_month_balance",
        F.last("ending_monthly_balance", ignorenulls=True).over(last_known_balance_window)
    )
    .withColumn(
        "previous_month_products",
        F.lag("active_products_count").over(behavior_window)
    )
)

## 3. Calculate the changes 

In [21]:
monthly_behavior_df = (
    monthly_behavior_df
    .withColumn(
        "balance_change",
        F.col("ending_monthly_balance")
        - F.col("previous_month_balance")
    )
    .withColumn(
        "product_change",
        F.col("active_products_count")
        - F.col("previous_month_products")
    )
)

In [22]:
monthly_behavior_df.show(20, truncate=False)

+--------+-----------+--------------+----------+----------------------+---------------------+----------------------+-----------------------+--------------+--------------+
|cust_key|customer_id|activity_month|is_churned|ending_monthly_balance|active_products_count|previous_month_balance|previous_month_products|balance_change|product_change|
+--------+-----------+--------------+----------+----------------------+---------------------+----------------------+-----------------------+--------------+--------------+
|65      |15567630   |2026-04-01    |true      |52124.9               |1                    |null                  |null                   |null          |null          |
|322     |15574167   |2025-12-01    |false     |88419.71              |1                    |null                  |null                   |null          |null          |
|434     |15576928   |2026-07-01    |false     |217.72                |1                    |null                  |null                   |null 

In [23]:
print("Rows where current balance is null:",
      monthly_behavior_df.filter(F.col("ending_monthly_balance").isNull()).count())

print("Rows where balance_change is still null (excluding true first-month rows):",
      monthly_behavior_df.filter(
          F.col("balance_change").isNull() & F.col("previous_month_balance").isNotNull()
      ).count())

Rows where current balance is null: 11076


Rows where balance_change is still null (excluding true first-month rows): 6387


## 4. Summarize each customer's overall trend
In this step, I will turn month by month values into one raw per customer indicates whether balance, products turning up or down and the percentage of their months showed a decline.
This achieved by:
1. Group by customer id, and is_churned
2. calculate average balance, average products
3. calculate the percentage by calculate the declining_balance_month over month with change

In [24]:
customer_behavior_summary_df = (
    monthly_behavior_df
    .filter(F.col("balance_change").isNotNull())
    .groupBy("customer_id", "is_churned")
    .agg(
        F.avg("balance_change").alias("avg_balance_change"),
        F.avg("product_change").alias("avg_product_change"),
        F.sum(
            F.when(F.col("balance_change") < 0, 1).otherwise(0)
        ).alias("declining_balance_months"),
        F.count("*").alias("months_with_change")
    )
    .withColumn(
        "pct_months_balance_declined",
        F.round(
            F.col("declining_balance_months") / F.col("months_with_change") * 100,
            2
        )
    )
)

## Compare Churned with non- churned
1. group by the is_churned
2. calculate the aggregation avg_balance, avg_product, avg_percentage

In [25]:
churn_behavior_comparison_df = (
    customer_behavior_summary_df
    .groupBy("is_churned")
    .agg(
        F.round(F.avg("avg_balance_change"), 2).alias("avg_monthly_balance_change"),
        F.round(F.avg("avg_product_change"), 2).alias("avg_monthly_product_change"),
        F.round(F.avg("pct_months_balance_declined"), 2).alias("avg_pct_months_balance_declined"),
        F.count("*").alias("num_customers")
    )
)

churn_behavior_comparison_df.show(truncate=False)

+----------+--------------------------+--------------------------+-------------------------------+-------------+
|is_churned|avg_monthly_balance_change|avg_monthly_product_change|avg_pct_months_balance_declined|num_customers|
+----------+--------------------------+--------------------------+-------------------------------+-------------+
|true      |-20574.85                 |0.24                      |47.43                          |609          |
|false     |-52371.77                 |0.22                      |47.88                          |3831         |
+----------+--------------------------+--------------------------+-------------------------------+-------------+



# Insight 2 — Support Friction vs. Churn
Analyze whether customers with higher support friction have higher churn rates.

Compare churn rates across customers with different levels of ticket volume, critical issues, and resolution time.

## 1. Aggregate support behavior by customer
Using the fact table joining with dim_customer calculate:
a. total number of tickets
b. total critical tickets
c. total resolution time
d. average monthly resolution time
All group by customer key, customer id, is churned.

In [26]:
support_customer_df = spark.sql("""
    SELECT
        c.cust_key,
        c.customer_id,
        c.is_churned,

        SUM(f.tickets_opened_count)
            AS total_tickets,

        SUM(f.critical_tickets_count)
            AS total_critical_tickets,

        SUM(f.total_resolution_time_hrs)
            AS total_resolution_time_hrs,

        AVG(f.total_resolution_time_hrs)
            AS avg_monthly_resolution_time_hrs

    FROM fact_customer_monthly_activity_df f

    JOIN dim_customer c
        ON f.cust_key = c.cust_key

    GROUP BY
        c.cust_key,
        c.customer_id,
        c.is_churned
""")

In [27]:
support_customer_df.show(20, truncate=False)

+--------+-----------+----------+-------------+----------------------+-------------------------+-------------------------------+
|cust_key|customer_id|is_churned|total_tickets|total_critical_tickets|total_resolution_time_hrs|avg_monthly_resolution_time_hrs|
+--------+-----------+----------+-------------+----------------------+-------------------------+-------------------------------+
|6266    |15722479   |false     |2            |0                     |38.900000000000006       |6.483333333333334              |
|927     |15589435   |false     |2            |0                     |104.5                    |34.833333333333336             |
|3528    |15654574   |false     |1            |0                     |16.0                     |5.333333333333333              |
|9624    |15806438   |true      |0            |0                     |0.0                      |0.0                            |
|7935    |15764153   |false     |0            |0                     |0.0                      |0

## 2. Calculate average resolution time per ticket
This to create the avarage resolution time per customer by:
1. create a column nammed avg_resolution_time_per_ticket
2. Calculate its value as the total resolution hours / total tickets if total tickets not equal zero

In [28]:
support_customer_df = (
    support_customer_df
    .withColumn(
        "avg_resolution_time_per_ticket",
        F.when(
            F.col("total_tickets") > 0,
            F.col("total_resolution_time_hrs")
            / F.col("total_tickets")
        ).otherwise(0)
    )
)

## 3. Compare churned with non churned

In [29]:
support_customer_df.groupBy("is_churned").agg(
    F.count("*").alias("customer_count"),

    F.avg("total_tickets")
        .alias("avg_total_tickets"),

    F.avg("total_critical_tickets")
        .alias("avg_total_critical_tickets"),

    F.avg("total_resolution_time_hrs")
        .alias("avg_total_resolution_time_hrs"),

    F.avg("avg_resolution_time_per_ticket")
        .alias("avg_resolution_time_per_ticket")
).show()

+----------+--------------+------------------+--------------------------+-----------------------------+------------------------------+
|is_churned|customer_count| avg_total_tickets|avg_total_critical_tickets|avg_total_resolution_time_hrs|avg_resolution_time_per_ticket|
+----------+--------------+------------------+--------------------------+-----------------------------+------------------------------+
|      true|          2025|0.6103703703703703|       0.09037037037037036|            462.9693333333333|             305.5804781893004|
|     false|          7919|0.6090415456497033|       0.08890011365071347|            676.1604116681395|             530.9683505914047|
+----------+--------------+------------------+--------------------------+-----------------------------+------------------------------+



## 4. Create support-friction groups
divide customers into groups:
    
    1. High Friction (At least 2 critical tickets OR average resolution time >= 24 hours)
    2. Medium Friction (At least 1 critical ticket OR average resolution time >= 12 hours)
    3. Low Friction (otherwise)

These numbers are assumptions:


In [30]:
support_customer_df = (
    support_customer_df
    .withColumn(
        "support_friction_group",
        F.when(
            (F.col("total_critical_tickets") >= 2) |
            (F.col("avg_resolution_time_per_ticket") >= 24),
            "High Friction"
        )
        .when(
            (F.col("total_critical_tickets") >= 1) |
            (F.col("avg_resolution_time_per_ticket") >= 12),
            "Medium Friction"
        )
        .otherwise("Low Friction")
    )
)

## 5. Calculate churn rate based by support group

In [31]:
support_customer_df.groupBy(
    "support_friction_group"
).agg(
    F.count("*").alias("customer_count"),

    F.sum(
        F.when(F.col("is_churned") == True, 1).otherwise(0)
    ).alias("churned_customers"),

    (
        F.sum(
            F.when(F.col("is_churned") == True, 1).otherwise(0)
        ) / F.count("*") * 100
    ).alias("churn_rate_percentage")
).orderBy(
    F.desc("churn_rate_percentage")
).show()

+----------------------+--------------+-----------------+---------------------+
|support_friction_group|customer_count|churned_customers|churn_rate_percentage|
+----------------------+--------------+-----------------+---------------------+
|       Medium Friction|          1225|              261|   21.306122448979593|
|          Low Friction|          6508|             1331|    20.45175169022741|
|         High Friction|          2211|              433|     19.5838986883763|
+----------------------+--------------+-----------------+---------------------+



# Insight 3: Marketing Offer Effectiveness

Is offer engagement associated with more stable balances and lower customer churn?

## 1. Aggregate offers by customer
From the join of the fact table with customer_dim:

    1. calculate the total offers receivedd
    2. calculate the total offers accepted
    3. calculate the average monthly balance

In [32]:
offer_customer_df = spark.sql("""
    SELECT
        c.cust_key,
        c.customer_id,
        c.is_churned,

        SUM(f.offers_received_count)
            AS total_offers_received,

        SUM(f.offers_accepted_count)
            AS total_offers_accepted,

        AVG(f.ending_monthly_balance)
            AS avg_monthly_balance

    FROM fact_customer_monthly_activity_df f

    JOIN dim_customer c
        ON f.cust_key = c.cust_key

    GROUP BY
        c.cust_key,
        c.customer_id,
        c.is_churned
""")

In [33]:
offer_customer_df.show(20, truncate=False)

+--------+-----------+----------+---------------------+---------------------+-------------------+
|cust_key|customer_id|is_churned|total_offers_received|total_offers_accepted|avg_monthly_balance|
+--------+-----------+----------+---------------------+---------------------+-------------------+
|6266    |15722479   |false     |3                    |2                    |478.3933333333334  |
|927     |15589435   |false     |1                    |0                    |981.17             |
|3528    |15654574   |false     |0                    |0                    |75314.43           |
|9624    |15806438   |true      |0                    |0                    |89952.18           |
|7935    |15764153   |false     |0                    |0                    |47491.535          |
|2371    |15625706   |true      |2                    |0                    |80357.735          |
|2101    |15619116   |false     |0                    |0                    |557.9              |
|9235    |15796343  

## 2. Calculate offer acceptance rate
The acceptance rate = total accepted over total received, as long as total received not equal zero

In [37]:
offer_customer_df = (
    offer_customer_df
    .withColumn(
        "offer_acceptance_rate",
        F.when(
            F.col("total_offers_received") > 0,
            F.col("total_offers_accepted")
            / F.col("total_offers_received")
        ).otherwise(0)
    )
)

## 3. Calculate balance stability
Calculate it by getting the the standard deviation of monthly balances.

In [36]:
offer_customer_df = spark.sql("""
    SELECT
        c.cust_key,
        c.customer_id,
        c.is_churned,

        SUM(f.offers_received_count)
            AS total_offers_received,

        SUM(f.offers_accepted_count)
            AS total_offers_accepted,

        AVG(f.ending_monthly_balance)
            AS avg_monthly_balance,

        STDDEV(f.ending_monthly_balance)
            AS balance_standard_deviation

    FROM fact_customer_monthly_activity_df f

    JOIN dim_customer c
        ON f.cust_key = c.cust_key

    GROUP BY
        c.cust_key,
        c.customer_id,
        c.is_churned
""")

## 4. Compare Churn cusomter with non churned

In [38]:
offer_customer_df.groupBy("is_churned").agg(
    F.count("*").alias("customer_count"),

    F.avg("total_offers_received")
        .alias("avg_offers_received"),

    F.avg("total_offers_accepted")
        .alias("avg_offers_accepted"),

    F.avg("offer_acceptance_rate")
        .alias("avg_offer_acceptance_rate"),

    F.avg("avg_monthly_balance")
        .alias("avg_monthly_balance"),

    F.avg("balance_standard_deviation")
        .alias("avg_balance_standard_deviation")
).show()

+----------+--------------+-------------------+-------------------+-------------------------+-------------------+------------------------------+
|is_churned|customer_count|avg_offers_received|avg_offers_accepted|avg_offer_acceptance_rate|avg_monthly_balance|avg_balance_standard_deviation|
+----------+--------------+-------------------+-------------------+-------------------------+-------------------+------------------------------+
|      true|          2025| 0.8133333333333334| 0.2474074074074074|      0.11377542621987072| 311688.56998026563|              544118.344399525|
|     false|          7919| 0.8122237656269731|  0.269857305215305|      0.12203455823546741|  397440.6080968843|             397420.6128998092|
+----------+--------------+-------------------+-------------------+-------------------------+-------------------+------------------------------+



## 5. Create offer-engagement groups
divide customers into groups:

    1. No Offers
    2. High Acceptance
    3. Low Acceptance

In [39]:
offer_customer_df = (
    offer_customer_df
    .withColumn(
        "offer_engagement_group",
        F.when(
            F.col("total_offers_received") == 0,
            "No Offers"
        )
        .when(
            F.col("offer_acceptance_rate") >= 0.50,
            "High Acceptance"
        )
        .when(
            F.col("offer_acceptance_rate") > 0,
            "Low Acceptance"
        )
        .otherwise("No Accepted Offers")
    )
)

## 6. Calculate churn rate by offer group

In [40]:
offer_customer_df.groupBy(
    "offer_engagement_group"
).agg(
    F.count("*").alias("customer_count"),

    F.sum(
        F.when(F.col("is_churned") == True, 1).otherwise(0)
    ).alias("churned_customers"),

    (
        F.sum(
            F.when(F.col("is_churned") == True, 1).otherwise(0)
        ) / F.count("*") * 100
    ).alias("churn_rate_percentage"),

    F.avg("avg_monthly_balance")
        .alias("avg_monthly_balance"),

    F.avg("balance_standard_deviation")
        .alias("avg_balance_standard_deviation")
).orderBy(
    F.desc("churn_rate_percentage")
).show()

+----------------------+--------------+-----------------+---------------------+-------------------+------------------------------+
|offer_engagement_group|customer_count|churned_customers|churn_rate_percentage|avg_monthly_balance|avg_balance_standard_deviation|
+----------------------+--------------+-----------------+---------------------+-------------------+------------------------------+
|    No Accepted Offers|          1755|              380|    21.65242165242165| 370347.99113119143|            430067.00087104324|
|             No Offers|          6243|             1266|    20.27871215761653| 400441.68690862524|             444845.9231032176|
|        Low Acceptance|           552|              111|   20.108695652173914| 228777.83757589554|             406683.2608467016|
|       High Acceptance|          1394|              268|   19.225251076040173| 360443.48939848266|            284544.80732543266|
+----------------------+--------------+-----------------+---------------------+----

# Insight 4: Customer Value Analysis / Simulated LTV
because my datasets do not contain actual transactions, revenue, or profit, we will describe it as Simulated Customer Lifetime Value (LTV) rather than actual financial CLV.

## 1. Aggregate customer value information
Calculate:

    Average monthly balance
    Maximum monthly balance
    Average number of active products
    Maximum number of active products
    Customer tenure
    Churn status

In [41]:
customer_value_df = spark.sql("""
    SELECT
        c.cust_key,
        c.customer_id,
        c.is_churned,
        c.tenure,

        AVG(f.ending_monthly_balance)
            AS avg_monthly_balance,

        MAX(f.ending_monthly_balance)
            AS max_monthly_balance,

        AVG(f.active_products_count)
            AS avg_active_products,

        MAX(f.active_products_count)
            AS max_active_products,

        COUNT(DISTINCT f.date_key)
            AS observed_active_months

    FROM fact_customer_monthly_activity_df f

    JOIN dim_customer c
        ON f.cust_key = c.cust_key

    GROUP BY
        c.cust_key,
        c.customer_id,
        c.is_churned,
        c.tenure
""")

## 2. Handle missing balance information
Identify whether the balance field is null or not and make a column with the boolean value

In [43]:
customer_value_df = (
    customer_value_df
    .withColumn(
        "balance_information_missing",
        F.when(
            F.col("avg_monthly_balance").isNull(),
            True
        ).otherwise(False)
    )
)

# 3. Calculate the balance score
As we have different scales, so we should normalize their values before combining them

The normalization will give me a value between 0 and 1, the value close to 0 means low average balance and the value close to 1 means high

### calculate the minimum and maximum balance:

In [44]:
balance_stats = customer_value_df.agg(
    F.min("avg_monthly_balance").alias("min_balance"),
    F.max("avg_monthly_balance").alias("max_balance")
).first()

min_balance = balance_stats["min_balance"]
max_balance = balance_stats["max_balance"]

### create the normalized balance score:

In [45]:
customer_value_df = (
    customer_value_df
    .withColumn(
        "balance_score",
        F.when(
            F.col("avg_monthly_balance").isNull(),
            0.0
        )
        .when(
            F.lit(max_balance) == F.lit(min_balance),
            0.0
        )
        .otherwise(
            (
                F.col("avg_monthly_balance") - F.lit(min_balance)
            )
            /
            (
                F.lit(max_balance) - F.lit(min_balance)
            )
        )
    )
)

## 4. Calculate the product usage score

In [46]:
customer_value_df = (
    customer_value_df
    .withColumn(
        "product_score",
        F.when(
            F.col("avg_active_products").isNull(),
            0.0
        )
        .otherwise(
            F.col("avg_active_products") / F.lit(4.0)
        )
    )
)

## 5. Calculate the tenure score

In [47]:
customer_value_df = (
    customer_value_df
    .withColumn(
        "tenure_score",
        F.when(
            F.col("tenure").isNull(),
            0.0
        )
        .otherwise(
            F.col("tenure") / F.lit(10.0)
        )
    )
)

## 6. Calculate the simulated LTV score
After having the balance, product usage and tenure, now we can calculate the LTV

The numbers are simulated

In [48]:
customer_value_df = (
    customer_value_df
    .withColumn(
        "simulated_ltv_score",
        (
            F.col("balance_score") * F.lit(0.50)
            +
            F.col("product_score") * F.lit(0.30)
            +
            F.col("tenure_score") * F.lit(0.20)
        )
    )
)

## 7. Create customer value groups
Devide the customer into groups:

    1. High value
    2. Medium value
    3. Low value

In [49]:
customer_value_df = (
    customer_value_df
    .withColumn(
        "customer_value_group",
        F.when(
            F.col("simulated_ltv_score") >= 0.67,
            "High Value"
        )
        .when(
            F.col("simulated_ltv_score") >= 0.34,
            "Medium Value"
        )
        .otherwise("Low Value")
    )
)

## 8. Display the customer-level results

In [50]:
customer_value_df.select(
    "cust_key",
    "customer_id",
    "is_churned",
    "tenure",
    "avg_monthly_balance",
    "avg_active_products",
    "simulated_ltv_score",
    "customer_value_group"
).show(truncate=False)

+--------+-----------+----------+------+-------------------+-------------------+-------------------+--------------------+
|cust_key|customer_id|is_churned|tenure|avg_monthly_balance|avg_active_products|simulated_ltv_score|customer_value_group|
+--------+-----------+----------+------+-------------------+-------------------+-------------------+--------------------+
|6716    |15733361   |true      |6     |63892.06           |0.3333333333333333 |0.14564833432874355|Low Value           |
|765     |15585595   |false     |1     |49707.7            |0.5                |0.05800440083341946|Low Value           |
|2080    |15618437   |false     |10    |18646.215          |1.0                |0.27518920944614456|Low Value           |
|9209    |15795737   |false     |3     |41222.975000000006 |1.0                |0.13541830346095332|Low Value           |
|6935    |15738662   |false     |9     |116980.51          |0.3333333333333333 |0.2061870407751281 |Low Value           |
|5454    |15701885   |fa

## 9. Calculate churn rate by value group

In [51]:
value_churn_df = (
    customer_value_df
    .groupBy("customer_value_group")
    .agg(
        F.count("*")
            .alias("customer_count"),

        F.sum(
            F.when(F.col("is_churned") == True, 1)
             .otherwise(0)
        )
        .alias("churned_customers"),

        F.avg(
            F.col("is_churned").cast("double")
        )
        .alias("churn_rate"),

        F.avg("avg_monthly_balance")
            .alias("average_balance"),

        F.avg("avg_active_products")
            .alias("average_active_products"),

        F.avg("tenure")
            .alias("average_tenure"),

        F.avg("simulated_ltv_score")
            .alias("average_simulated_ltv_score")
    )
    .orderBy("customer_value_group")
)

In [52]:
value_churn_df.show(truncate=False)

+--------------------+--------------+-----------------+-------------------+--------------------+-----------------------+-----------------+---------------------------+
|customer_value_group|customer_count|churned_customers|churn_rate         |average_balance     |average_active_products|average_tenure   |average_simulated_ltv_score|
+--------------------+--------------+-----------------+-------------------+--------------------+-----------------------+-----------------+---------------------------+
|High Value          |4             |0                |0.0                |4.601615932750001E7 |1.0625                 |7.75             |0.701629022451324          |
|Low Value           |9857          |2015             |0.20442325251090596|148161.91518077222  |0.7544276004000059     |5.008420411890027|0.15824630099953885        |
|Medium Value        |83            |10               |0.12048192771084337|2.5576231812148593E7|0.9022088353413655     |5.614457831325301|0.4394854884043678         

# Store The business insights in HDFS

### The paths

In [53]:
INSIGHTS_BASE_PATH = (
    "/user/student/Capstone_project/gold/insights"
)
INSIGHT_1_PATH = (
    f"{INSIGHTS_BASE_PATH}/early_warning_churn"
)

INSIGHT_2_PATH = (
    f"{INSIGHTS_BASE_PATH}/support_friction_churn"
)

INSIGHT_3_PATH = (
    f"{INSIGHTS_BASE_PATH}/offer_effectiveness"
)

INSIGHT_4_PATH = (
    f"{INSIGHTS_BASE_PATH}/customer_value_analysis"
)

In [ ]:
monthly_behavior_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(INSIGHT_1_PATH)

support_customer_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(INSIGHT_2_PATH)


offer_customer_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(INSIGHT_3_PATH)


customer_value_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(INSIGHT_4_PATH)

customer_value_df.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save(INSIGHT_4_PATH)

In [ ]:
spark.sql(f"""
    create table if not exists
    {HIVE_DB}.insight_early_warning_churn
    using parquet
    location '{INSIGHT_1_PATH}'
""")

spark.sql(f"""
    create table if not exists
    {HIVE_DB}.insight_support_friction_churn
    using parquet
    location '{INSIGHT_2_PATH}'
""")

spark.sql(f"""
    create table if not exists
    {HIVE_DB}.insight_offer_effectiveness
    using parquet
    location '{INSIGHT_3_PATH}'
""")

spark.sql(f"""
    create table if not exists
    {HIVE_DB}.insight_customer_value
    using parquet
    location '{INSIGHT_4_PATH}'
""")